# Google Search Ranking & Discoverability Capstone
## Lane: Refresh / Content Opportunity Scoring

**Research question:** Can signals available at the end of a month rank content items that are likely to experience a meaningful impression decline in the following month?

This notebook uses the gated FlyRank warehouse release to build month-aligned examples, compares a transparent rule baseline with Logistic Regression and Random Forest, selects a model using validation months only, and evaluates it on a May 2026 feature window with June 2026 as the outcome window.

The output is a decision-support queue for human review. It is not evidence that content refreshes cause traffic gains and does not attempt to explain Google's ranking algorithm.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/uzairrateef-rgb/my-flyrank-internship.git"
REPO_DIR = Path("/content/my-flyrank-internship") if IN_COLAB else Path.cwd()

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "duckdb", "pandas", "numpy", "scikit-learn", "matplotlib"],
        check=True,
    )

ROOT = Path.cwd()
print("Repository root:", ROOT)

In [ ]:
import json
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "Add a read-only Hugging Face token as a Colab Secret named HF_TOKEN "
        "(or set HF_TOKEN in your local environment). Do not paste it into this notebook."
    )

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
del HF_TOKEN

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
print("Authenticated to the gated release; the token was not printed or saved.")

## 1. Question

The decision is which pages an editor should inspect first during a limited refresh-review cycle. A false positive wastes review time; a false negative leaves a potentially declining page unchecked.

The label is an explicitly defined next-month proxy: an eligible content item is positive when next-month GSC impressions are at most 80% of current-month impressions. The item must have at least 100 impressions in the feature month and valid GSC records for the previous, feature, and outcome months.

The research tests whether a model ranks this next-month decline proxy better than a simple rule that combines current visibility with recent momentum.

## 2. Data and public-safe contract

- **Release:** FlyRank/internship-warehouse, gated Hugging Face Parquet release.
- **Table:** fact_content_daily_performance, aggregated from daily rows into client × content × month.
- **Feature windows:** previous and current complete calendar months.
- **Outcome window:** the following complete calendar month.
- **Split:** train feature months June 2025–January 2026; validation feature months February–April 2026; final test features May 2026 with June 2026 outcomes.
- **Data eligibility:** only records with gsc_data_available IS TRUE; NULL/false flags are not interpreted as zero activity. Items need a row in all three adjacent months and at least 100 current-month impressions.
- **Excluded from predictors:** content/client IDs, trend labels, trend_pct, outcome-month data, raw queries, URLs, domains, client names, and GA4/AI-referral fields. IDs are only used to join the same content across months.

This selection favors continuously observed content and can underrepresent new, removed, or intermittently tracked pages. The final report must state this limitation.

In [ ]:
# Start with a lightweight metadata probe, then perform the full aggregation once.
# The derived cache lives under data/processed/, which .gitignore excludes.
# Do not commit the cache: it contains internal pseudonymous item keys.
CACHE_PATH = ROOT / "data" / "processed" / "capstone_monthly_gsc_v1.parquet"
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

probe = con.sql(f"""
SELECT COUNT(*) AS rows_in_march,
       MIN(report_date) AS min_date,
       MAX(report_date) AS max_date
FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print("Metadata probe for March 2026 (preflight):")
display(probe)

month_starts = pd.date_range("2025-05-01", "2026-06-01", freq="MS")
partition_paths = [
    f"{WAREHOUSE}/fact_content_daily_performance/month={month:%Y-%m}/*.parquet"
    for month in month_starts
]
partition_list_sql = "[" + ", ".join("'" + p.replace("'", "''") + "'" for p in partition_paths) + "]"

if CACHE_PATH.exists():
    print("Using previously generated local aggregate cache:", CACHE_PATH)
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE monthly_gsc AS
    SELECT * FROM read_parquet('{CACHE_PATH.as_posix()}')
    """)
else:
    print("No local cache found. Running the full warehouse aggregation once; later reruns reuse the ignored cache.")
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE monthly_gsc AS
    SELECT
        client_hash_id,
        content_hash_id,
        DATE_TRUNC('month', report_date)::DATE AS month,
        SUM(COALESCE(gsc_impressions, 0))::DOUBLE AS impressions,
        SUM(COALESCE(gsc_clicks, 0))::DOUBLE AS clicks,
        SUM(CASE WHEN gsc_avg_position > 0 AND COALESCE(gsc_impressions, 0) > 0
                 THEN gsc_avg_position * gsc_impressions ELSE 0 END)
          / NULLIF(SUM(CASE WHEN gsc_avg_position > 0 AND COALESCE(gsc_impressions, 0) > 0
                            THEN gsc_impressions ELSE 0 END), 0) AS avg_position,
        COUNT(*) FILTER (WHERE COALESCE(gsc_impressions, 0) > 0) AS active_days
    FROM read_parquet({partition_list_sql}, union_by_name = true)
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id, DATE_TRUNC('month', report_date)::DATE
    """)
    con.execute(f"COPY monthly_gsc TO '{CACHE_PATH.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    print("Wrote ignored local cache. Do not add it to Git:", CACHE_PATH)

monthly_audit = con.sql("""
SELECT month, COUNT(*) AS content_client_month_rows,
       COUNT(DISTINCT content_hash_id) AS content_items,
       SUM(impressions) AS impressions
FROM monthly_gsc
GROUP BY month ORDER BY month
""").df()
print("Monthly aggregation rows:", con.sql("SELECT COUNT(*) FROM monthly_gsc").fetchone()[0])
display(monthly_audit)
print("Partitions covered:", len(partition_paths), "from May 2025 through June 2026")

In [ ]:
# Build rows using only previous/current-month features and the following month's label.
examples = con.sql("""
SELECT
    curr.month AS feature_month,
    curr.client_hash_id,
    curr.content_hash_id,
    prev.impressions AS impressions_prev_month,
    prev.clicks AS clicks_prev_month,
    prev.avg_position AS position_prev_month,
    prev.active_days AS active_days_prev_month,
    curr.impressions AS impressions_current_month,
    curr.clicks AS clicks_current_month,
    curr.avg_position AS position_current_month,
    curr.active_days AS active_days_current_month,
    future.impressions AS impressions_next_month,
    CASE WHEN future.impressions <= 0.80 * curr.impressions THEN 1 ELSE 0 END
        AS decline_20pct_next_month
FROM monthly_gsc curr
JOIN monthly_gsc prev
  ON prev.client_hash_id = curr.client_hash_id
 AND prev.content_hash_id = curr.content_hash_id
 AND prev.month = curr.month - INTERVAL '1 month'
JOIN monthly_gsc future
  ON future.client_hash_id = curr.client_hash_id
 AND future.content_hash_id = curr.content_hash_id
 AND future.month = curr.month + INTERVAL '1 month'
WHERE curr.month >= DATE '2025-06-01'
  AND curr.month <= DATE '2026-05-01'
  AND curr.impressions >= 100
""").df()

examples["feature_month"] = pd.to_datetime(examples["feature_month"])
examples["ctr_prev_month"] = 100 * examples["clicks_prev_month"] / examples["impressions_prev_month"].replace(0, np.nan)
examples["ctr_current_month"] = 100 * examples["clicks_current_month"] / examples["impressions_current_month"].replace(0, np.nan)
examples["impressions_change_pct"] = 100 * (
    examples["impressions_current_month"] - examples["impressions_prev_month"]
) / examples["impressions_prev_month"].replace(0, np.nan)
examples["position_change"] = examples["position_current_month"] - examples["position_prev_month"]

TARGET = "decline_20pct_next_month"
FEATURES = [
    "impressions_prev_month", "clicks_prev_month", "position_prev_month",
    "active_days_prev_month", "impressions_current_month", "clicks_current_month",
    "position_current_month", "active_days_current_month", "ctr_prev_month",
    "ctr_current_month", "impressions_change_pct", "position_change",
]
FORBIDDEN = {
    TARGET, "impressions_next_month", "trend_direction", "trend_pct",
    "client_hash_id", "content_hash_id", "feature_month",
}
assert not (set(FEATURES) & FORBIDDEN), "Leakage or identifier found among features."
assert examples[TARGET].isin([0, 1]).all()
assert examples[FEATURES].select_dtypes(include="number").shape[1] == len(FEATURES)

print("Eligible examples:", f"{len(examples):,}")
print("Feature months:", examples.feature_month.min().date(), "to", examples.feature_month.max().date())
print("Number of model features:", len(FEATURES))
print("No IDs, target columns, trend labels, or outcome-month metrics are predictors.")

## 3. Methodology and validation

### Feature timing and leakage controls

The predictor columns contain only GSC signals from the previous and current full calendar months. CTR and month-over-month changes are derived only from those same past windows. The next-month impression total is used only to construct the target and is excluded from model features. Trend fields and pseudonymous IDs are excluded from predictors.

### Baseline

The transparent rule score assigns 50% weight to the percentile rank of log current-month impressions and 50% to recent momentum risk (more negative month-over-month impression change gets a higher score).

### Candidate models

Logistic Regression and Random Forest estimate the probability of the next-month decline proxy. The model is chosen by validation Precision@50, then average precision and ROC AUC as tie-breakers. A deterministic sample cap of 250,000 training rows limits Colab runtime; evaluation cohorts remain unsampled.

### Split

Train feature months end in January 2026, validation covers February–April 2026, and May 2026 is the final holdout with June 2026 outcomes. The same content may appear in earlier months and later months, which reflects prediction for an existing portfolio; this does not measure transfer to unseen clients. The final holdout is not used to select the model.

In [ ]:
def precision_at_k(y_true, scores, k=50):
    y = np.asarray(y_true, dtype=int)
    score = np.asarray(scores, dtype=float)
    n = min(int(k), len(y))
    if n == 0:
        return float("nan")
    return float(y[np.argsort(-score)[:n]].mean())

def ranking_metrics(frame, scores):
    y = frame[TARGET].astype(int).to_numpy()
    scores = np.asarray(scores, dtype=float)
    metrics = {
        "n": int(len(frame)),
        "decline_base_rate": float(y.mean()),
        "precision_at_20": precision_at_k(y, scores, 20),
        "precision_at_50": precision_at_k(y, scores, 50),
        "precision_at_100": precision_at_k(y, scores, 100),
    }
    if len(np.unique(y)) == 2:
        metrics["roc_auc"] = float(roc_auc_score(y, scores))
        metrics["average_precision"] = float(average_precision_score(y, scores))
    else:
        metrics["roc_auc"] = None
        metrics["average_precision"] = None
    return metrics

def baseline_score(frame):
    visibility = np.log1p(frame["impressions_current_month"].clip(lower=0)).rank(pct=True)
    momentum = frame["impressions_change_pct"].replace([np.inf, -np.inf], np.nan).fillna(0)
    momentum_risk = (-momentum).rank(pct=True)
    return (0.50 * visibility + 0.50 * momentum_risk).to_numpy()

train = examples[examples.feature_month <= pd.Timestamp("2026-01-01")].copy()
valid = examples[examples.feature_month.between(pd.Timestamp("2026-02-01"), pd.Timestamp("2026-04-01"))].copy()
test = examples[examples.feature_month == pd.Timestamp("2026-05-01")].copy()

assert len(train) and len(valid) and len(test), "A time split is empty; verify data coverage."
assert train.feature_month.max() < valid.feature_month.min()
assert valid.feature_month.max() < test.feature_month.min()
assert train[TARGET].nunique() == 2 and valid[TARGET].nunique() == 2

RANDOM_STATE = 42
TRAIN_SAMPLE_CAP = 250_000

def build_model(name):
    if name == "logistic_regression":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=500, random_state=RANDOM_STATE)),
        ])
    if name == "random_forest":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=100, max_depth=8, min_samples_leaf=50,
                class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE
            )),
        ])
    raise ValueError(name)

train_fit = train.sample(n=min(TRAIN_SAMPLE_CAP, len(train)), random_state=RANDOM_STATE)
X_train = train_fit[FEATURES].replace([np.inf, -np.inf], np.nan)
y_train = train_fit[TARGET].astype(int)
X_valid = valid[FEATURES].replace([np.inf, -np.inf], np.nan)
validation_results = {}

for name in ["logistic_regression", "random_forest"]:
    candidate = build_model(name)
    candidate.fit(X_train, y_train)
    scores = candidate.predict_proba(X_valid)[:, 1]
    validation_results[name] = ranking_metrics(valid, scores)

validation_table = pd.DataFrame(validation_results).T[
    ["n", "decline_base_rate", "precision_at_20", "precision_at_50",
     "precision_at_100", "roc_auc", "average_precision"]
]
display(validation_table)

def metric_for_sort(metrics, key):
    value = metrics.get(key)
    return float(value) if value is not None and np.isfinite(value) else -1.0

selected_model_name = max(
    validation_results,
    key=lambda name: (
        metric_for_sort(validation_results[name], "precision_at_50"),
        metric_for_sort(validation_results[name], "average_precision"),
        metric_for_sort(validation_results[name], "roc_auc"),
    ),
)
print("Model selected on validation only:", selected_model_name)
print("Training examples sampled for fit:", f"{len(train_fit):,}", "of", f"{len(train):,}")

## 4. Results — model versus baseline

The next cell refits the validation-selected model on examples with feature months through April 2026, then evaluates it on May 2026 features and June 2026 outcomes. The rule baseline and selected model use exactly the same final holdout rows. Precision@K is reported beside the target base rate, plus ROC AUC and average precision. These values are calculated by this notebook, not typed into the report by hand.

In [ ]:
# Refit using only pre-holdout labels (through May 2026 outcome month).
fit_all = examples[examples.feature_month <= pd.Timestamp("2026-04-01")].copy()
fit_sample = fit_all.sample(n=min(TRAIN_SAMPLE_CAP, len(fit_all)), random_state=RANDOM_STATE)
final_model = build_model(selected_model_name)
final_model.fit(
    fit_sample[FEATURES].replace([np.inf, -np.inf], np.nan),
    fit_sample[TARGET].astype(int),
)

test_scores = final_model.predict_proba(test[FEATURES].replace([np.inf, -np.inf], np.nan))[:, 1]
baseline_scores = baseline_score(test)
model_metrics = ranking_metrics(test, test_scores)
baseline_metrics = ranking_metrics(test, baseline_scores)

results_table = pd.DataFrame({
    "Transparent baseline": baseline_metrics,
    "Selected model": model_metrics,
}).T[["n", "decline_base_rate", "precision_at_20", "precision_at_50",
     "precision_at_100", "roc_auc", "average_precision"]]
display(results_table)

base_rate = model_metrics["decline_base_rate"]
p50_lift_base_rate = model_metrics["precision_at_50"] / base_rate if base_rate > 0 else None
p50_lift_baseline = (
    model_metrics["precision_at_50"] / baseline_metrics["precision_at_50"]
    if baseline_metrics["precision_at_50"] > 0 else None
)
print(f"Holdout decline base rate: {base_rate:.3f}")
print("Precision@50 lift over base rate:", f"{p50_lift_base_rate:.2f}x" if p50_lift_base_rate is not None else "undefined")
print("Precision@50 lift over baseline:", f"{p50_lift_baseline:.2f}x" if p50_lift_baseline is not None else "undefined")

# Aggregate error counts at a 0.50 probability threshold; do not print row identifiers.
test_y = test[TARGET].astype(int).to_numpy()
test_pred = (test_scores >= 0.5).astype(int)
error_counts = {
    "true_positives": int(((test_pred == 1) & (test_y == 1)).sum()),
    "false_positives": int(((test_pred == 1) & (test_y == 0)).sum()),
    "false_negatives": int(((test_pred == 0) & (test_y == 1)).sum()),
    "true_negatives": int(((test_pred == 0) & (test_y == 0)).sum()),
}
display(pd.DataFrame([{"error_type": k, "count": v} for k, v in error_counts.items()]))
print("False positives are candidates that did not meet the defined decline threshold; false negatives are qualifying declines missed at threshold 0.50.")

## 5. Limitations

- The label is a defined 20% month-on-month decline proxy, not a verified editorial decision.
- One final outcome month cannot establish performance across all seasons or market conditions.
- Items without GSC coverage in all three adjacent months are excluded; this creates a persistent-observation selection effect.
- Monthly aggregation hides daily volatility. Seasonal cycles, demand changes, migrations, or measurement changes may look like content deterioration.
- This evaluates future-month ranking for an existing tracked portfolio, not generalization to unseen clients.
- Feature importance describes predictive use, not causes. The study does not test whether refreshing content improves clicks, impressions, or position.

Use the words observed, measured, directional, associated, and decision-support. Do not claim causal refresh impact or that the model predicts Google's algorithm.

## 6. Ranked recommendations

1. **Inspect visible content with worsening momentum.** Verify seasonality, demand shifts, and tracking before recommending a refresh.
2. **Review CTR in context.** Compare only adequate-volume pages with similar intent and position; do not treat small-sample CTR as conclusive.
3. **Investigate persistent position deterioration.** Check indexing, SERP changes, intent mismatch, internal links, and competitor changes.
4. **Review factual freshness and task completion.** Update stale claims, examples, screenshots, references, or internal links only when an actual user need exists.
5. **Monitor ambiguous items.** Do not force edits on noisy or low-signal rows.

A human editor must approve all changes. Never delete, redirect, or bulk-publish content based solely on a score, and never keyword-stuff content to satisfy the model.

## 7. Paper artifacts

The following aggregate-only outputs are generated in the next cell:

- work/outputs/capstone_metrics.json — method, date windows, selected model, validation metrics, final holdout metrics, and aggregate error counts.
- docs/paper-assets/decline_rate_by_month.svg — next-month decline share by feature month.
- docs/paper-assets/model_vs_baseline.svg — model versus baseline Precision@K on the same final holdout, with the base rate.
- docs/paper-assets/feature_importance.svg — selected model's feature importance or coefficient magnitude.

No row-level queue, content/client identifiers, raw queries, or daily exports are written.

In [ ]:
OUTPUT_DIR = ROOT / "work" / "outputs"
PAPER_ASSET_DIR = ROOT / "docs" / "paper-assets"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAPER_ASSET_DIR.mkdir(parents=True, exist_ok=True)

# Aggregate monthly outcome shares; this chart includes the holdout only after final scoring is complete.
monthly_labels = examples.groupby("feature_month")[TARGET].agg(["size", "mean"]).reset_index()
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(monthly_labels["feature_month"], monthly_labels["mean"], marker="o")
for x_value, y_value, n_value in zip(monthly_labels["feature_month"], monthly_labels["mean"], monthly_labels["size"]):
    ax.annotate(f"n={int(n_value):,}", (x_value, y_value), xytext=(0, 8), textcoords="offset points", ha="center", fontsize=7)
ax.set_title("Observed next-month decline rate by feature month (n shown at each point)")
ax.set_xlabel("Feature month (label is the following month)")
ax.set_ylabel("Share with ≥20% impression decline")
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.grid(axis="y", alpha=0.25)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(PAPER_ASSET_DIR / "decline_rate_by_month.svg", format="svg", bbox_inches="tight")
plt.show()
plt.close(fig)

# Model vs baseline on the exact same holdout, with the test base rate.
ks = [20, 50, 100]
model_p = [model_metrics[f"precision_at_{k}"] for k in ks]
baseline_p = [baseline_metrics[f"precision_at_{k}"] for k in ks]
positions = np.arange(len(ks))
fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.plot(positions, model_p, marker="o", label=f"Selected model ({selected_model_name})")
ax.plot(positions, baseline_p, marker="o", label="Transparent baseline")
ax.axhline(base_rate, linestyle="--", label=f"Holdout base rate ({base_rate:.1%})")
ax.set_xticks(positions, [f"Precision@{k}" for k in ks])
ax.set_ylabel("Precision")
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_ylim(0, min(1.0, max(max(model_p), max(baseline_p), base_rate) * 1.2 + 0.03))
ax.set_title(f"Ranking quality on May → June 2026 holdout (n={len(test):,})")
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(PAPER_ASSET_DIR / "model_vs_baseline.svg", format="svg", bbox_inches="tight")
plt.show()
plt.close(fig)

estimator = final_model.named_steps["model"]
if hasattr(estimator, "feature_importances_"):
    importance_values = np.asarray(estimator.feature_importances_, dtype=float)
    importance_method = "Random Forest impurity importance (not causal)"
else:
    importance_values = np.abs(np.asarray(estimator.coef_).ravel())
    importance_method = "Absolute standardized Logistic Regression coefficient (not causal)"
importance_frame = pd.DataFrame({"feature": FEATURES, "importance": importance_values})
importance_top = importance_frame.sort_values("importance", ascending=False).head(10)
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(importance_top.sort_values("importance")["feature"], importance_top.sort_values("importance")["importance"])
ax.set_title(f"Features used by the selected model (fit n={len(fit_sample):,})")
ax.set_xlabel(importance_method)
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(PAPER_ASSET_DIR / "feature_importance.svg", format="svg", bbox_inches="tight")
plt.show()
plt.close(fig)

metrics_payload = {
    "study": "Refresh / Content Opportunity Scoring",
    "data_release": "FlyRank/internship-warehouse",
    "source_table": "fact_content_daily_performance",
    "grain": "client-content-month aggregate",
    "windows": {
        "train_feature_months": [str(train.feature_month.min().date()), str(train.feature_month.max().date())],
        "validation_feature_months": [str(valid.feature_month.min().date()), str(valid.feature_month.max().date())],
        "final_test_feature_month": "2026-05",
        "final_test_outcome_month": "2026-06",
    },
    "label": "next-month impressions <= 80% of current-month impressions",
    "eligibility": "GSC IS TRUE for previous/current/outcome month; at least 100 current-month impressions",
    "features": FEATURES,
    "excluded_from_features": sorted(FORBIDDEN),
    "random_seed": RANDOM_STATE,
    "training_sample_cap": TRAIN_SAMPLE_CAP,
    "train_rows_available": int(len(train)),
    "train_rows_sampled_for_validation_fit": int(len(train_fit)),
    "validation_rows": int(len(valid)),
    "test_rows": int(len(test)),
    "selected_model_on_validation": selected_model_name,
    "validation_metrics": validation_results,
    "final_test": {
        "base_rate": base_rate,
        "model": model_metrics,
        "baseline": baseline_metrics,
        "precision_at_50_lift_over_base_rate": p50_lift_base_rate,
        "precision_at_50_lift_over_baseline": p50_lift_baseline,
        "error_counts_at_probability_0_50": error_counts,
    },
    "feature_importance_method": importance_method,
    "feature_importance": [
        {"feature": str(row.feature), "importance": float(row.importance)}
        for row in importance_top.itertuples(index=False)
    ],
    "limitations": [
        "one final holdout outcome month",
        "decline is a defined proxy, not editorial ground truth",
        "only continuously observed GSC items with at least 100 current-month impressions are eligible",
        "no causal refresh impact is estimated",
    ],
}
metrics_path = OUTPUT_DIR / "capstone_metrics.json"
metrics_path.write_text(json.dumps(metrics_payload, indent=2))
print("Wrote aggregate metrics:", metrics_path)
print("Wrote paper figures:", PAPER_ASSET_DIR)
print("Do not publish row-level queues or pseudonymous identifiers.")

# Bundle only aggregate/public-safe artifacts for easy download from Colab.
# ZIP is gitignored; extract and commit only the JSON + three SVGs, never the ZIP/cache.
from zipfile import ZipFile, ZIP_DEFLATED

artifact_zip = OUTPUT_DIR / "capstone_public_artifacts.zip"
public_artifacts = [
    (metrics_path, "work/outputs/capstone_metrics.json"),
    (PAPER_ASSET_DIR / "decline_rate_by_month.svg", "docs/paper-assets/decline_rate_by_month.svg"),
    (PAPER_ASSET_DIR / "model_vs_baseline.svg", "docs/paper-assets/model_vs_baseline.svg"),
    (PAPER_ASSET_DIR / "feature_importance.svg", "docs/paper-assets/feature_importance.svg"),
]
with ZipFile(artifact_zip, "w", compression=ZIP_DEFLATED) as bundle:
    for source_path, archive_path in public_artifacts:
        bundle.write(source_path, arcname=archive_path)

print("Created public-safe artifact bundle:", artifact_zip)
if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(artifact_zip))
    except Exception as exc:
        print("Automatic download was unavailable; download this file from the Colab Files sidebar:", artifact_zip)
        print(type(exc).__name__, str(exc))

## 8. Reproducibility

- Fixed random seed: 42.
- Hugging Face token is read from Colab Secrets or an environment variable and is never stored in the notebook.
- The full monthly Parquet partitions are aggregated in DuckDB; only aggregate metrics and figures are exported.
- Features are restricted to previous/current month; outcome-month impressions are used only for the label.
- Model selection uses validation months; the final May → June 2026 holdout is evaluated after selection.
- Baseline and selected model are compared on the same final cohort.
- The final metrics receipt is work/outputs/capstone_metrics.json. The output charts are SVGs in docs/paper-assets.
- Re-run from a fresh runtime and ensure the JSON and figures agree before updating the public paper.

Repository: https://github.com/uzairrateef-rgb/my-flyrank-internship

## 9. Acknowledgments & data credit

Built on the [FlyRank ML Internship dataset](https://flyrank.ai).

## ML-12 — five-minute demo outline

- **0:00–0:45 — Decision:** content review prioritization and the cost of wrong calls.
- **0:45–1:30 — Data:** warehouse table, monthly grain, eligibility, and privacy exclusions.
- **1:30–2:30 — Method:** future-month label, baseline, candidate models, validation windows, and final holdout.
- **2:30–3:30 — Results:** model versus baseline Precision@K next to the base rate.
- **3:30–4:30 — Recommendations:** how editors use a reasoned queue without automatic publishing.
- **4:30–5:00 — Limits:** proxy label, one-month holdout, coverage bias, seasonality, and no causal evidence.

In [ ]:
# ML-12 — social-post cut and three-sentence employer-facing summary, populated from this run.
social_post = (
    "I built a time-aware content refresh opportunity-scoring workflow using the FlyRank ML "
    "Internship warehouse. On a held-out May→June 2026 evaluation, the selected "
    f"{selected_model_name} model achieved Precision@50={model_metrics['precision_at_50']:.3f}, "
    f"versus {baseline_metrics['precision_at_50']:.3f} for a transparent baseline, against "
    f"a decline base rate of {base_rate:.3f}. The score is a review aid, not a promise of search "
    "gains or evidence that a refresh causes traffic growth."
)
employer_summary = (
    "I built a reproducible content refresh prioritization workflow on FlyRank's gated, "
    "pseudonymized daily search-performance warehouse, with month-aligned features and a "
    "next-month decline proxy. "
    f"I compared Logistic Regression and Random Forest with a transparent baseline using "
    f"validation months and a final May→June 2026 holdout; selected-model Precision@50 was "
    f"{model_metrics['precision_at_50']:.3f} versus {baseline_metrics['precision_at_50']:.3f} "
    f"for the baseline, against a {base_rate:.3f} holdout base rate. "
    "I documented leakage controls, public-data safeguards, reasoned editorial actions, and "
    "the limits imposed by seasonality, coverage, and observational evidence."
)
print("SOCIAL POST\n\n" + social_post)
print("\nEMPLOYER SUMMARY (3 sentences)\n\n" + employer_summary)

## Final submission checklist

- [ ] Run Runtime → Run all in Google Colab using a read-only HF token stored as a secret.
- [ ] Confirm the splits are chronologically ordered and May features predict June outcomes.
- [ ] Review the final metrics table and base rate. Do not substitute old or guessed results.
- [ ] Commit work/notebooks/capstone.ipynb, work/outputs/capstone_metrics.json, and the three aggregate figures.
- [ ] Update work/capstone_report.md and docs/index.html with this exact run's metrics; remove provisional wording only after the rerun succeeds.
- [ ] Enable GitHub Pages from main /docs, verify the public page in a private window, then replace submission/paper_url.txt with the exact live URL as its only line.
- [ ] Never commit data exports, row-level queues, client/content IDs, raw queries, domains, URLs, or credentials.

The study measures ranking quality for a defined next-month decline proxy. It does not demonstrate the effect of content refreshes or prove anything about Google's algorithm.